In [1]:
%pip install --upgrade numpy pandas scikit-learn

  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/b9/c5/fc1b368f303087d20e8c9bf3d6ceb186263cfac0ade735cd938538bea839/pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata
  Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.3-cp312-cp312-win_amd64.whl (9.8 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.1
    Uninstalling pandas-2.2.1:
      Successfully uninstalled pandas-2.2.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# =============================================================================
# 1. Configuración de Rutas
# =============================================================================
RUTA_CLIMA = Path("../Datos/Staging/clima_limpio.csv")
RUTA_ACCIDENTES = Path("../Datos/Raw/accidentes.csv")
RUTA_RAW_STAGING = Path("../Datos/Staging/accidentes_staging_enriquecido.csv") 

TARGET_COL = "hubo_accidente"
RANDOM_STATE = 42
TEST_SIZE = 0.20

# =============================================================================
# 2. Paso Ligero 1: Indexar el Target en frío para hacer el Split Estratificado
# =============================================================================
print("--> Fase 1: Extrayendo esqueleto del universo para calcular el Split...")
acc_base = pd.read_csv(RUTA_ACCIDENTES, parse_dates=["TW"])
acc_base["KEY_BUSQUEDA"] = acc_base["BARRIO"].astype(str) + "_" + acc_base["TW"].astype(str)
accidentes_set = set(acc_base["KEY_BUSQUEDA"])
del acc_base

# ¡CORREGIDO!: Se añade chunksize=1_000_000 para que 'chunk' sea un DataFrame real en cada iteración
esqueleto_partes = []
for chunk in pd.read_csv(RUTA_CLIMA, usecols=["BARRIO", "TW"], parse_dates=["TW"], chunksize=1_000_000):
    keys_chunk = chunk["BARRIO"].astype(str) + "_" + chunk["TW"].astype(str)
    chunk[TARGET_COL] = np.where(keys_chunk.isin(accidentes_set), 1, 0).astype("uint8")
    esqueleto_partes.append(chunk)

df_esqueleto = pd.concat(esqueleto_partes, ignore_index=True)
del esqueleto_partes
print(f"   Universo detectado: {len(df_esqueleto):,} filas.")

# Creamos el split estratégico de índices basados en el esqueleto ultra-liviano
indices_train, indices_test = train_test_split(
    df_esqueleto.index, 
    test_size=TEST_SIZE, 
    stratify=df_esqueleto[TARGET_COL], 
    random_state=RANDOM_STATE
)

# Guardamos las estadísticas para la pregunta 3 del taller antes de borrar el esqueleto
pos_c = int(df_esqueleto[TARGET_COL].sum())
total_c = len(df_esqueleto)
print(f"   [RESPUESTA TALLER]: Proporción global de clase positiva (accidentes): {(pos_c / total_c) * 100:.4f}%")

# Convertimos los índices a sets para búsquedas O(1) veloces en el bucle principal
set_indices_train = set(indices_train)
del df_esqueleto, indices_train, indices_test # Liberación masiva de RAM

# =============================================================================
# 3. Carga de tu Feature Engineering desde Staging
# =============================================================================
print("\n--> Fase 2: Cargando tus variables enriquecidas...")
raw_staging = pd.read_csv(RUTA_RAW_STAGING, parse_dates=["TW"])

columnas_features_agg = [
    "hora_dia", "dia_semana", "mes_num", "es_fin_semana", "es_festivo", 
    "hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos", "dia_ano_sin", 
    "dia_ano_cos", "comuna_ext", "prom_accidentes_historico_barrio"
]

agg_dict = {"Lat": "median", "Lon": "median"}
for col in columnas_features_agg:
    if col in raw_staging.columns:
        agg_dict[col] = "first"

features_agregadas = raw_staging.groupby(["BARRIO", "TW"]).agg(agg_dict).reset_index()
features_agregadas["BARRIO"] = features_agregadas["BARRIO"].astype("category")
features_agregadas["Lat"] = features_agregadas["Lat"].astype("float32")
features_agregadas["Lon"] = features_agregadas["Lon"].astype("float32")
del raw_staging

# =============================================================================
# 4. Paso 2: Re-procesar por bloques distribuyendo directamente a Train y Test
# =============================================================================
print("\n--> Fase 3: Construyendo matrices X e y finales separadas en RAM...")

DTYPES_CLIMA = {
    "BARRIO": "category", "temperature": "float32", "apparentTemperature": "float32",
    "humidity": "float32", "precipIntensity": "float32", "precipProbability": "float32",
    "windSpeed": "float32", "cloudCover": "float32", "visibility": "float32"
}

X_train_partes, y_train_partes = [], []
X_test_partes, y_test_partes = [], []

fila_actual_global = 0

for chunk in pd.read_csv(RUTA_CLIMA, chunksize=1_000_000, parse_dates=["TW"], dtype=DTYPES_CLIMA):
    chunk = chunk.dropna(subset=["temperature", "humidity", "windSpeed"])
    chunk["BARRIO"] = chunk["BARRIO"].astype("category")
    
    # Pegamos tus variables enriquecidas
    chunk_enriquecido = pd.merge(chunk, features_agregadas, on=["BARRIO", "TW"], how="left")
    
    # Calculamos el target local
    keys_chunk = chunk_enriquecido["BARRIO"].astype(str) + "_" + chunk_enriquecido["TW"].astype(str)
    chunk_enriquecido[TARGET_COL] = np.where(keys_chunk.isin(accidentes_set), 1, 0).astype("uint8")
    
    # Re-creamos el mapeo de índices correspondiente a este chunk específico
    indices_del_chunk = np.arange(fila_actual_global, fila_actual_global + len(chunk_enriquecido))
    fila_actual_global += len(chunk_enriquecido)
    
    # Máscaras booleanas de división
    es_train = np.isin(indices_del_chunk, list(set_indices_train))
    
    # Limpieza final de columnas predictoras
    if "BARRIO" in chunk_enriquecido.columns:
        chunk_enriquecido["BARRIO"] = chunk_enriquecido["BARRIO"].cat.codes
    
    chunk_predictores = chunk_enriquecido.drop(columns=["TW", "summary", "icon", "fecha_dt", TARGET_COL], errors="ignore").fillna(0)
    chunk_target = chunk_enriquecido[TARGET_COL]
    
    # Separación inmediata sin acumular en una sola lista gigante
    X_train_partes.append(chunk_predictores[es_train])
    y_train_partes.append(chunk_target[es_train])
    X_test_partes.append(chunk_predictores[~es_train])
    y_test_partes.append(chunk_target[~es_train])

del features_agregadas, accidentes_set, set_indices_train

# Concatemos los conjuntos por separado (ocupan la mitad de espacio que la tabla completa unificada)
print("--> Consolidando conjuntos de datos finales...")
X_train = pd.concat(X_train_partes, ignore_index=True)
y_train = pd.concat(y_train_partes, ignore_index=True)
X_test = pd.concat(X_test_partes, ignore_index=True)
y_test = pd.concat(y_test_partes, ignore_index=True)

del X_train_partes, y_train_partes, X_test_partes, y_test_partes
print(f"¡Split seguro completado! Registros Entrenamiento: {len(X_train):,}, Registros Prueba: {len(X_test):,}")

# =============================================================================
# 5. Reporte de Distribución de Clases (Código del Profesor)
# =============================================================================
def get_binary_class_counts(split_y):
    neg_c = int((split_y == 0).sum())
    pos_c = int((split_y == 1).sum())
    total = len(split_y)
    return {"negative_count": neg_c, "positive_count": pos_c, "positive_rate": pos_c / total if total > 0 else 0}

split_rows = []
for split_name, split_y in {"train": y_train, "test": y_test}.items():
    stats = get_binary_class_counts(split_y)
    split_rows.append({"split": split_name, **stats, "positive_rate_percent": stats["positive_rate"] * 100})

try:
    display(pd.DataFrame(split_rows))
except NameError:
    print(pd.DataFrame(split_rows))

--> Fase 1: Extrayendo esqueleto del universo para calcular el Split...


MemoryError: Unable to allocate 512. KiB for an array with shape (65536,) and data type int64